In [1]:
import sys
sys.path.append('../')

import pandas as pd
from src.agents.graph import build_fraud_graph



Loading model bundle...
Loading SHAP explainer...


In [2]:
app = build_fraud_graph()
print("graph compiled successfully")

Graph compiled successfully
graph compiled successfully


In [3]:
X_test = pd.read_parquet("data/processed/X_test.parquet")
Y_test = pd.read_parquet("data/processed/Y_test.parquet").iloc[:,0]



In [4]:
import numpy as np
import joblib

In [5]:
bundle = joblib.load('data/models/xgboost_production.pkl')
model  = bundle['model']
probs = model.predict_proba(X_test)[:,1]

In [7]:
high_risk_idx = probs.argmax()
low_risk_idx = probs.argmin()



In [8]:
def run_transaction(idx, label=""):
    tx_data = X_test.iloc[idx].to_dict()
    tx_id = f"TX_{idx:06d}"


    print(f"\n{'='*55}")
    print(f"Running {label} transaction: {tx_id}")
    print(f"{'='*55}")

    initial_state={
        "transaction_id": tx_id,
        "transaction_data": tx_data,
        "fraud_probability": None,
        "risk_level": None,
        "shap_explanation":None,
        "explanation_text": None,
        "decision": None,
        "policy_reasoning": None,
        "requires_human": None,
        "final_report":None,
        "processing_errors":[],
    }

    result = app.invoke(initial_state)
    report = result["final_report"]

    print(f"\n── Final Report ──")
    print(f"Transaction:  {report['transaction_id']}")
    print(f"Probability:  {report['fraud_probability']:.4f}")
    print(f"Risk level:   {report['risk_level']}")
    print(f"Decision:     {report['decision']}")
    print(f"Human review: {report['requires_human']}")
    print(f"\nPolicy reasoning:")
    print(f"  {report['policy_reasoning']}")
    print(f"\nExplanation:")
    print(f"  {report['explanation']}")

    return report





In [9]:
high_report = run_transaction(high_risk_idx, "HIGH RISK")
low_report = run_transaction(low_risk_idx, 'LOW RISK')


Running HIGH RISK transaction: TX_085892
[RiskScorer] tx=TX_085892 prob=1.0000 risk=high
[Explainer DEBUG] explanation keys: dict_keys(['base_value', 'prediction', 'top_features'])
[Explainer DEBUG] first feature item: {'feature': 'V258', 'shap_value': 1.221718, 'actual_value': 4.0}
[Explainer] tx=TX_085892 top_feature=V258
[Policy] tx=TX_085892 decision=deny human=True
[HumanReview] tx = TX_085892queued for analyst review
[REPORT] tx = TX_085892decision= deny complete
[REPORT] tx=TX_085892decision=deny complete

── Final Report ──
Transaction:  TX_085892
Probability:  1.0000
Risk level:   high
Decision:     deny
Human review: True

Policy reasoning:
  Transaction denied. Fraud probability 1.0000exceeds threshold  0.2161. Flagged for human review.
[QUEUED FOR HUMAN REVIEW]

Explanation:
  Fraud probability: 1.0000
Key factors:
- V258 (value: 4.0)increased fraud score by  1.222
- C14 (value: 0.0)increased fraud score by  1.045
- TransactionAmt (value: 300.0)increased fraud score by  0.